# Queries

### Imports and connection setup

In [1]:
from neo4j import GraphDatabase

# db connection
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "DataMan2024")

### Query 1
Which Pokémon are the most used and with which teratype?
(Note that: each Pokémon is considered only once, specifically with the Teratype corresponding to the highest number of uses.)


In [2]:
# Query 1
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session(database="neo4j") as session:
        
        # Define the query
        command = """
        MATCH (team:Team)-[:USE]->(p:Pokemon)-[r:HAS_TERATYPE]->(t:Type)
        WITH p.pokemon_name AS pokemon_name, t.type_name AS teratype,
            COUNT(DISTINCT team.player_name) AS usage_count
        ORDER BY usage_count DESC
        WITH pokemon_name, 
            COLLECT({teratype: teratype, usage_count: usage_count}) AS teratypes
        RETURN pokemon_name, 
            teratypes[0].teratype AS most_used_teratype, 
            teratypes[0].usage_count AS max_usage_count
        ORDER BY max_usage_count DESC
        LIMIT 10;
        """
        
        # Execute the query and get the records
        result = session.run(command)
        
        # Print the result with formatting
        for record in result:
            # Accessing the values by index, since it's returned as a list
            pokemon_name = record[0]  # First value in the list
            most_used_teratype = record[1]  # Second value in the list
            max_usage_count = record[2]  # Third value in the list
            
            # Format the output
            print(f"pokemon_name: {pokemon_name}, "
                  f"most_used_teratype: {most_used_teratype}, "
                  f"max_usage_count: {max_usage_count}")
            

pokemon_name: urshifu-rapid-strike, most_used_teratype: grass, max_usage_count: 398
pokemon_name: rillaboom, most_used_teratype: grass, max_usage_count: 303
pokemon_name: raging-bolt, most_used_teratype: water, max_usage_count: 242
pokemon_name: incineroar, most_used_teratype: grass, max_usage_count: 218
pokemon_name: farigiraf, most_used_teratype: grass, max_usage_count: 210
pokemon_name: iron-hands, most_used_teratype: grass, max_usage_count: 181
pokemon_name: calyrex-shadow, most_used_teratype: grass, max_usage_count: 155
pokemon_name: calyrex-ice, most_used_teratype: grass, max_usage_count: 151
pokemon_name: miraidon, most_used_teratype: electric, max_usage_count: 126
pokemon_name: chi-yu, most_used_teratype: grass, max_usage_count: 117


### Query 2
Difference in mean between competitive and regular gameplay pokemons.
(Note that each Pokémon should be considered only once, even if it has multiple Teratypes and "stats" refers to the sum of their individual statistics.)


In [3]:
# Query 2
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session(database="neo4j") as session:
        
        # Define the query
        command = """
        MATCH (p:Pokemon)-[r:HAS_TERATYPE]->(t:Type)
        WITH ROUND(AVG(p.hp + p.attack + p.defense + p.special_attack + p.special_defense + p.speed), 2) AS avg_competitive_stats
        MATCH (p:Pokemon)
        WITH ROUND(AVG(p.hp + p.attack + p.defense + p.special_attack + p.special_defense + p.speed), 2) AS avg_regular_stats, avg_competitive_stats
        RETURN avg_regular_stats, avg_competitive_stats, 
        ROUND(avg_regular_stats - avg_competitive_stats, 2) AS stats_difference;
        """
        
        # Execute the query and get the records
        result = session.run(command)
        
        # Print the result with formatting
        for record in result:
            # Accessing the values by index, since it's returned as a list
            avg_regular_stas = record[0]  # First value in the list
            avg_competitive_stas = record[1]  # Second value in the list
            stas_difference = record[2]  # Third value in the list
            
            # Format the output
            print(f"avg_regular_stas: {avg_regular_stas}, "
                  f"avg_competitive_stas: {avg_competitive_stas}, "
                  f"stas_difference: {stas_difference}")


avg_regular_stas: 445.86, avg_competitive_stas: 553.43, stas_difference: -107.57


### Query 3
Differences in type/dual type between Pokémon used in competitive play and in regular gameplay
(Note that: the pokemon used for the comparison are the one from the first query)


In [4]:
# Query 3
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session(database="neo4j") as session:
        
        # Define the query
        command = """
        MATCH (team:Team)-[:USE]->(p:Pokemon)-[r:HAS_TERATYPE]->(t:Type)
        WITH p.pokemon_name AS pokemon_name, t.type_name AS teratype, 
        COUNT(DISTINCT r.player_name) AS usage_count
        ORDER BY pokemon_name, usage_count DESC
        WITH pokemon_name, COLLECT({teratype: teratype, usage_count: usage_count}) AS teratype_usage
        MATCH (p:Pokemon {pokemon_name: pokemon_name})-[:HAS_TYPE]->(type:Type)
        WITH pokemon_name, teratype_usage[0].teratype AS most_used_teratype, teratype_usage[0].usage_count AS usage_count, COLLECT(type.type_name) AS types
        RETURN pokemon_name, most_used_teratype, types
        ORDER BY usage_count DESC
        LIMIT 10;

        """
        
        # Execute the query and get the records
        result = session.run(command)
        
        # Print the result with formatting
        for record in result:
            # Accessing the values by index, since it's returned as a list
            pokemon_name = record[0]  # First value in the list
            most_used_teratype = record[1]  # Second value in the list
            max_usage_count = record[2]  # Third value in the list
            
            # Format the output
            print(f"pokemon_name: {pokemon_name}, "
                  f"most_used_teratype: {most_used_teratype}, "
                  f"max_usage_count: {max_usage_count}") 



pokemon_name: rillaboom, most_used_teratype: fire, max_usage_count: ['grass']
pokemon_name: raging-bolt, most_used_teratype: electric, max_usage_count: ['electric', 'dragon']
pokemon_name: incineroar, most_used_teratype: ghost, max_usage_count: ['fire', 'dark']
pokemon_name: urshifu-rapid-strike, most_used_teratype: water, max_usage_count: ['fighting', 'water']
pokemon_name: ogerpon-hearthflame-mask, most_used_teratype: fire, max_usage_count: ['grass', 'fire']
pokemon_name: miraidon, most_used_teratype: fairy, max_usage_count: ['electric', 'dragon']
pokemon_name: ogerpon-cornerstone-mask, most_used_teratype: rock, max_usage_count: ['grass', 'rock']
pokemon_name: farigiraf, most_used_teratype: water, max_usage_count: ['normal', 'psychic']
pokemon_name: iron-hands, most_used_teratype: grass, max_usage_count: ['fighting', 'electric']
pokemon_name: calyrex-ice, most_used_teratype: fire, max_usage_count: ['psychic', 'ice']


### Close connection and session

In [5]:
# close connection and session
driver = GraphDatabase.driver(URI, auth=AUTH)
session = driver.session(database="neo4j")

session.close()
driver.close()